# Modelado del Número de Puntos Anotados en Partidos de la NBA
# Procesos de Poisson, Markov, Estacionariedad y Exponencial

**Materia:** Procesos Estocásticos  
**Proyecto Final**

---

## 1. Introducción

En el baloncesto profesional de la NBA, cada partido es una realización de un fenómeno aleatorio complejo. Este proyecto aplica tres grandes temas de procesos estocásticos al análisis de datos reales:

- **Proceso de Poisson:** Modelar los puntos anotados por partido como una distribución Poisson(λ), incluyendo análisis homogéneo vs no homogéneo.
- **Distribución Exponencial:** Analizar los tiempos entre "eventos" (partidos de altos puntos) y verificar si siguen una distribución exponencial.
- **Cadenas de Markov:** Modelar resultados W/L como una cadena de Markov de 2 estados con matriz de transición, distribución estacionaria y predicción.
- **Estacionariedad:** Pruebas formales (ADF, KPSS) sobre la serie de puntos.

## 2. Marco Teórico

### 2.1 Distribución de Poisson
$$P(X = k) = \frac{\lambda^k e^{-\lambda}}{k!}, \quad k = 0, 1, 2, \dots$$

Propiedad fundamental: $E[X] = \text{Var}(X) = \lambda$

### 2.2 Distribución Exponencial
Si los eventos ocurren según un proceso de Poisson con tasa λ, los tiempos entre eventos siguen una distribución exponencial: $T \sim \text{Exp}(\lambda)$, con $E[T] = 1/\lambda$.

### 2.3 Cadenas de Markov
Una cadena de Markov es un proceso estocástico donde el estado futuro solo depende del estado presente: $P(X_{n+1} = j \mid X_n = i) = P_{ij}$. La distribución estacionaria π satisface $\pi = \pi P$.

### 2.4 Estacionariedad
Un proceso es estacionario en sentido débil si su media y varianza son constantes en el tiempo. Se verifica con las pruebas ADF (H₀: raíz unitaria) y KPSS (H₀: estacionariedad).

## 3. Tecnologías Utilizadas
- `nba_api` — Extracción de datos oficiales de NBA.com
- `pandas` — Manipulación y limpieza de datos
- `numpy` — Cálculos numéricos y simulación
- `matplotlib` — Visualización de datos
- `scipy.stats` — Distribuciones y pruebas estadísticas
- `statsmodels` — Pruebas de estacionariedad (ADF, KPSS)

In [1]:
import os
import sys
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
from scipy import stats
from scipy.stats import poisson

%matplotlib inline
plt.rcParams.update({
    "figure.figsize": (12, 6),
    "figure.dpi": 100,
    "font.size": 12,
    "axes.titlesize": 14,
    "axes.labelsize": 12,
    "figure.facecolor": "white",
    "axes.facecolor": "#f8f8f8",
    "axes.grid": True,
    "grid.alpha": 0.3,
})

PROJECT_ROOT = os.path.dirname(os.path.abspath("."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.clean import clean
from src.analysis import (
    estadistica_descriptiva, verificar_propiedad_poisson,
    poisson_pmf_teorica,
    prueba_estacionariedad, analizar_poisson_no_homogeneo,
    identificar_eventos_y_tiempos,
    graficar_histograma_comparativo, graficar_qq_poisson,
    graficar_cdf_escalonada, graficar_media_varianza_por_temporada,
    graficar_qq_exponencial,
    resumen_estadistico, imprimir_resumen
)
from src.markov import (
    construir_matriz_transicion, distribucion_estacionaria,
    predecir_siguiente, graficar_transiciones
)

print("Todas las librerias cargadas correctamente.")


Todas las librerias cargadas correctamente.


---
## 4. Selección del Equipo y Carga de Datos

Analizamos **Golden State Warriors** con datos de las temporadas 2023-24 y 2024-25 (2 temporadas).

Si ya tienes los datos extraídos, se usarán los archivos locales. Para extraer desde cero, cambia `SKIP_EXTRACT = False`.

In [2]:
EQUIPO = 'warriors'
NOMBRE_EQUIPO = 'Golden State Warriors'
SEASONS = ['2023-24', '2024-25']

print(f'Equipo seleccionado: {NOMBRE_EQUIPO}')
print(f'Temporadas a analizar: {len(SEASONS)} ({SEASONS[0]} a {SEASONS[-1]})')

df_clean, clean_path = clean(
    team_name=EQUIPO,
    project_root=PROJECT_ROOT,
    seasons=SEASONS,
)

print(f'\nShape del dataset limpio: {df_clean.shape}')
print(f'Columnas: {df_clean.columns.tolist()}')
df_clean[['fecha', 'puntos', 'resultado', 'enfrentamiento']].head(10)

Equipo seleccionado: Golden State Warriors
Temporadas a analizar: 2 (2023-24 a 2024-25)

  LIMPIEZA Y PREPARACIÓN DE DATOS

  Cargando datos desde: /Users/eduardonava/Desktop/PF_Procesos/proyecto_estocasticos_nba/data/raw/warriors_game_logs_raw.csv
  Registros cargados: 875
  Filtrado a temporadas ['2023-24', '2024-25']: 164 registros
  Datos procesados guardados en: /Users/eduardonava/Desktop/PF_Procesos/proyecto_estocasticos_nba/data/processed/warriors_game_logs_clean.csv

Shape del dataset limpio: (164, 20)
Columnas: ['id_equipo', 'equipo', 'id_partido', 'fecha', 'enfrentamiento', 'resultado', 'puntos', 'tiros_campo_convertidos', 'tiros_campo_intentados', 'porcentaje_tiros_campo', 'triples_convertidos', 'triples_intentados', 'porcentaje_triples', 'libres_convertidos', 'libres_intentados', 'porcentaje_libres', 'asistencias', 'rebotes', 'perdidas', 'mas_menos']


,fecha,puntos,resultado,enfrentamiento
0,2023-10-24,104,L,GSW vs. PHX
1,2023-10-27,122,W,GSW @ SAC
2,2023-10-29,106,W,GSW @ HOU
3,2023-10-30,130,W,GSW @ NOP
4,2023-11-01,102,W,GSW vs. SAC
5,2023-11-03,141,W,GSW @ OKC
6,2023-11-05,104,L,GSW @ CLE
7,2023-11-06,120,W,GSW @ DET
8,2023-11-08,105,L,GSW @ DEN
9,2023-11-11,110,L,GSW vs. CLE


---
## 5. Análisis Exploratorio de Datos (EDA)

Exploramos la distribución de puntos anotados con estadísticas descriptivas y visualizaciones iniciales.

In [3]:
puntos = df_clean['puntos'].values.astype(float)
n_partidos = len(puntos)

print(f'Total de partidos analizados: {n_partidos}')
print(f'\nEstadisticas descriptivas de PTS:')
print(df_clean['puntos'].describe())
print(f'\nModa: {df_clean["puntos"].mode().values[0]}')
print(f'Asimetria: {stats.skew(puntos):.3f}')
print(f'Curtosis: {stats.kurtosis(puntos):.3f}')

Total de partidos analizados: 164

Estadisticas descriptivas de PTS:
count    164.000000
mean     115.780488
std       12.773654
min       85.000000
25%      105.750000
50%      116.000000
75%      125.000000
max      148.000000
Name: puntos, dtype: float64

Moda: 104
Asimetria: -0.044
Curtosis: -0.413


In [4]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

ax = axes[0]
ax.hist(puntos, bins=25, density=True, alpha=0.7, color='#2196F3',
        edgecolor='white', linewidth=0.5, label='Densidad observada')
ax.axvline(np.mean(puntos), color='#D32F2F', linestyle='--', linewidth=2,
           label=f'Media = {np.mean(puntos):.1f}')
ax.axvline(np.median(puntos), color='#4CAF50', linestyle='-.', linewidth=2,
           label=f'Mediana = {np.median(puntos):.1f}')
ax.set_xlabel('Puntos por partido')
ax.set_ylabel('Densidad')
ax.set_title(f'Distribucion de Puntos \u2014 {NOMBRE_EQUIPO}')
ax.legend()

ax = axes[1]
ax.boxplot(puntos, vert=True, patch_artist=True,
           boxprops=dict(facecolor='#2196F3', alpha=0.6),
           medianprops=dict(color='#D32F2F', linewidth=2))
ax.set_ylabel('Puntos por partido')
ax.set_title(f'Diagrama de Caja \u2014 {NOMBRE_EQUIPO}')
ax.set_xticklabels([])

plt.tight_layout()
os.makedirs('plots', exist_ok=True)
plt.savefig('plots/eda_inicial.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 6. Estadística Descriptiva Formal

Calculamos las estadísticas descriptivas usando nuestro módulo de análisis.

In [5]:
stats_dict = estadistica_descriptiva(puntos)
lambda_est = stats_dict['lambda_estimado']

print(f'{"Metrica":<35} {"Valor":>10}')
print('-' * 47)
for k, v in stats_dict.items():
    if isinstance(v, float):
        print(f'{k:<35} {v:>10.3f}')
    else:
        print(f'{k:<35} {v:>10}')

print(f'\nLambda estimado (media muestral): {lambda_est:.3f}')

Metrica                                  Valor
-----------------------------------------------
n                                          164
media                                  115.780
varianza                               162.171
desviacion_estandar                     12.735
lambda_estimado                        115.780
minimo                                  85.000
maximo                                 148.000
mediana                                116.000
moda                                   104.000
asimetria                               -0.044
curtosis                                -0.413
ratio_varianza_media                     1.401

Lambda estimado (media muestral): 115.780


---
## 7. Verificación de la Propiedad: $E[X] = Var(X) = \lambda$

Una condición necesaria para que los datos sigan una Poisson es que media y varianza sean aproximadamente iguales. La razón $\text{Var}/E[X] \approx 1$ indica equidispersión.

In [6]:
propiedad = verificar_propiedad_poisson(stats_dict)

fig, ax = plt.subplots(figsize=(8, 5))
metricas = ['Media', 'Varianza']
valores = [propiedad['media'], propiedad['varianza']]
colores = ['#2196F3', '#FF9800']

bars = ax.bar(metricas, valores, color=colores, edgecolor='white', linewidth=1.5, width=0.4)
for bar, val in zip(bars, valores):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{val:.1f}', ha='center', va='bottom', fontweight='bold', fontsize=13)

ax.set_ylabel('Valor')
ax.set_title(f'Comparacion Media vs Varianza \u2014 {NOMBRE_EQUIPO}\n'
             f'Razon Var/E[X] = {propiedad["ratio_varianza_media"]:.4f}')
ax.set_ylim(0, max(valores) * 1.2)

ax.text(0.5, 0.95, f'Diagnostico: {propiedad["diagnostico"]}',
        transform=ax.transAxes, ha='center', fontsize=11,
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

plt.tight_layout()
plt.savefig('plots/propiedad_media_varianza.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nMedia (E[X]):           {propiedad["media"]:.2f}')
print(f'Varianza (Var[X]):       {propiedad["varianza"]:.2f}')
print(f'Razon Var/E[X]:         {propiedad["ratio_varianza_media"]:.4f}')
print(f'Diferencia relativa:    {propiedad["diferencia_relativa"]:.4f}')
print(f'\n{propiedad["diagnostico"]}')


Media (E[X]):           115.78
Varianza (Var[X]):       162.17
Razon Var/E[X]:         1.4007
Diferencia relativa:    0.4007

DESVIACIÓN SIGNIFICATIVA: Los datos muestran sobredispersión. Poisson puede no ser adecuado.


---
## 8. Ajuste de la Distribución de Poisson

Estimamos $\hat{\lambda} = \bar{x}$ por máxima verosimilitud y superponemos la PMF teórica sobre el histograma real.

In [7]:
x_pmf, y_pmf = poisson_pmf_teorica(lambda_est)

fig, ax = plt.subplots(figsize=(12, 6))
min_val = int(min(puntos.min(), lambda_est - 4 * np.sqrt(lambda_est)))
max_val = int(max(puntos.max(), lambda_est + 4 * np.sqrt(lambda_est)))
bins = np.arange(min_val - 0.5, max_val + 1.5, 1)

ax.hist(puntos, bins=bins, density=True, alpha=0.65, color='#2196F3',
        edgecolor='white', linewidth=0.5, label=f'Datos reales (n={n_partidos})')
ax.plot(x_pmf, y_pmf, 'o-', color='#D32F2F', linewidth=2.5, markersize=5,
        label=f'Poisson teorica ($\\lambda$ = {lambda_est:.2f})')
ax.axvline(lambda_est, color='#D32F2F', linestyle='--', alpha=0.4,
           label=f'$\\lambda$ = {lambda_est:.2f}')

ax.text(0.98, 0.95,
        f'$\\lambda$ estimado = {lambda_est:.2f}\n'
        f'n = {n_partidos} partidos\n'
        f'Media real = {np.mean(puntos):.2f}\n'
        f'Varianza real = {np.var(puntos):.2f}',
        transform=ax.transAxes, fontsize=10, verticalalignment='top',
        horizontalalignment='right',
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.85))

ax.set_xlabel('Puntos por partido (k)')
ax.set_ylabel('$P(X = k)$')
ax.set_title(f'Ajuste de Distribucion de Poisson \u2014 {NOMBRE_EQUIPO}')
ax.legend(loc='upper right')
ax.set_xlim(min_val, max_val)

plt.tight_layout()
plt.savefig('plots/ajuste_poisson.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 9. Función de Distribución Acumulada (CDF)

Comparamos la CDF empírica escalonada de los datos reales con la CDF teórica de Poisson. Mientras más se traslapen las curvas, mejor es el ajuste del modelo.

In [8]:
p_sorted = np.sort(puntos)
n = len(puntos)
cdf_empirica = np.arange(1, n + 1) / n

x_theo = np.arange(int(p_sorted[0]), int(p_sorted[-1]) + 1)
cdf_teorica = poisson.cdf(x_theo, mu=lambda_est)

max_diff = np.max(np.abs(
    np.searchsorted(p_sorted, x_theo, side='right') / n - cdf_teorica
))

fig, ax = plt.subplots(figsize=(10, 7))

ax.step(p_sorted, cdf_empirica, where='post', color='#2196F3',
        linewidth=2, label='CDF empirica (datos reales)')
ax.plot(x_theo, cdf_teorica, 'o--', color='#D32F2F', linewidth=2,
        markersize=5, label=f'CDF teorica Poisson ($\\lambda$={lambda_est:.1f})')

ax.set_xlabel('Puntos por partido')
ax.set_ylabel('Probabilidad acumulada')
ax.set_title(f'Funcion de Distribucion Acumulada (CDF) \u2014 {NOMBRE_EQUIPO}')
ax.legend(loc='lower right')

ax.text(0.02, 0.95,
        'La linea roja punteada es la CDF teorica de Poisson.\n'
        'Los escalones azules son la CDF empirica de los datos.\n'
        'Mientras mas se traslapen, mejor es el ajuste.',
        transform=ax.transAxes, fontsize=9, verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.85))

ax.text(0.98, 0.20,
        f'$\\lambda$ = {lambda_est:.2f}\n'
        f'n = {n} partidos\n'
        f'Diferencia maxima = {max_diff:.4f}',
        transform=ax.transAxes, fontsize=10, verticalalignment='top',
        horizontalalignment='right',
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

plt.tight_layout()
plt.savefig('plots/cdf_poisson.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 10. Q-Q Plot Poisson

Comparamos cuantiles observados vs cuantiles teóricos de Poisson. Si los puntos caen sobre la línea y=x, los datos se distribuyen como una Poisson.

In [9]:
fig, ax = plt.subplots(figsize=(8, 8))

n = len(puntos)
theoretical_quantiles = poisson.ppf((np.arange(1, n + 1) - 0.5) / n, mu=lambda_est)

ax.scatter(theoretical_quantiles, p_sorted, alpha=0.5, color='#2196F3',
           edgecolors='white', linewidth=0.3, s=50)
ax.plot(theoretical_quantiles, theoretical_quantiles, '--', color='#D32F2F',
        linewidth=2, label='Linea de referencia (y = x)')

ax.set_xlabel('Cuantiles teoricos (Poisson)')
ax.set_ylabel('Cuantiles observados')
ax.set_title(f'Q-Q Plot: Cuantiles Observados vs Teoricos Poisson')
ax.legend(loc='upper left')

ax.text(0.02, 0.95,
        'Si los puntos caen sobre la linea roja y=x,\n'
        'los datos se distribuyen como una Poisson.\n'
        'Desviaciones en las colas indican diferencias.',
        transform=ax.transAxes, fontsize=9, verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.85))

ax.text(0.98, 0.10,
        f'$\\lambda$ = {lambda_est:.2f}\n'
        f'n = {n} partidos',
        transform=ax.transAxes, fontsize=10, verticalalignment='top',
        horizontalalignment='right',
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

plt.tight_layout()
plt.savefig('plots/qq_plot_poisson_nb.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 11. Pruebas Formales de Estacionariedad

Un proceso de Poisson homogéneo requiere que $\lambda$ sea constante en el tiempo. Aplicamos dos pruebas complementarias:

- **ADF (Augmented Dickey-Fuller):** H₀ = la serie tiene raíz unitaria (no estacionaria).
- **KPSS:** H₀ = la serie es estacionaria.

Resultados consistentes:
- ADF rechaza H₀ y KPSS no rechaza H₀ → Serie **estacionaria**.
- ADF no rechaza H₀ y KPSS rechaza H₀ → Serie **no estacionaria**.
- Ambas rechazan o ambas no rechazan → Resultados **no concluyentes**.

In [10]:
estacionariedad = prueba_estacionariedad(puntos)

if estacionariedad['adf_estadistico'] is not None:
    print('--- Prueba ADF (Augmented Dickey-Fuller) ---')
    print(f'  Estadistico: {estacionariedad["adf_estadistico"]:.4f}')
    print(f'  P-valor:     {estacionariedad["adf_p_valor"]:.4f}')
    print(f'  Conclusion:  {estacionariedad["adf_conclusion"]}')
    print()
    print('--- Prueba KPSS ---')
    print(f'  Estadistico: {estacionariedad["kpss_estadistico"]:.4f}')
    print(f'  P-valor:     {estacionariedad["kpss_p_valor"]:.4f}')
    print(f'  Conclusion:  {estacionariedad["kpss_conclusion"]}')

    if ('ESTACIONARIA' in estacionariedad['adf_conclusion'] and
        'ESTACIONARIA' in estacionariedad['kpss_conclusion']):
        print('\n>>> Ambas pruebas indican que la serie es ESTACIONARIA.')
        print('>>> Lambda puede considerarse constante en el tiempo.')
else:
    print('statsmodels no disponible. Instala con: pip install statsmodels')

--- Prueba ADF (Augmented Dickey-Fuller) ---
  Estadistico: -3.9835
  P-valor:     0.0015
  Conclusion:  Serie ESTACIONARIA (rechaza H0 de raiz unitaria)

--- Prueba KPSS ---
  Estadistico: 0.4442
  P-valor:     0.0581
  Conclusion:  Serie ESTACIONARIA (no rechaza H0)

>>> Ambas pruebas indican que la serie es ESTACIONARIA.
>>> Lambda puede considerarse constante en el tiempo.


---
## 12. Estabilidad de $\lambda$ por Temporada

Calculamos media y varianza por temporada para verificar visualmente que $\lambda$ se mantiene estable.

In [11]:
df_temp = df_clean.copy()
df_temp['temporada'] = df_temp['fecha'].apply(
    lambda d: f'{d.year}-{str(d.year+1)[-2:]}' if d.month >= 10
    else f'{d.year-1}-{str(d.year)[-2:]}'
)

agg = df_temp.groupby('temporada')['puntos'].agg(['mean', 'var', 'count'])
agg = agg[agg['count'] >= 5]

fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(agg))

ax.bar(x - 0.15, agg['mean'], 0.3, label='Media', color='#2196F3',
       edgecolor='white', linewidth=0.5)
ax.bar(x + 0.15, agg['var'], 0.3, label='Varianza', color='#FF9800',
       edgecolor='white', linewidth=0.5)
ax.axhline(df_clean['puntos'].mean(), color='#D32F2F', linestyle=':',
           linewidth=2, label=f'Media global = {df_clean["puntos"].mean():.1f}')

ratio_global = df_clean['puntos'].var() / df_clean['puntos'].mean()

ax.set_xticks(x)
ax.set_xticklabels(agg.index, rotation=0, ha='center', fontsize=11)
ax.set_xlabel('Temporada')
ax.set_ylabel('Puntos')
ax.set_title(f'Estabilidad de $\\lambda$ por Temporada \u2014 {NOMBRE_EQUIPO}')
ax.legend(loc='upper left')

ax.text(0.98, 0.95,
        'En una Poisson, E[X] = Var(X) = $\\lambda$.\n'
        'Si las barras son similares y las lineas\n'
        'son horizontales, $\\lambda$ es estable.',
        transform=ax.transAxes, fontsize=9, verticalalignment='top',
        horizontalalignment='right',
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.85))

ax.text(0.02, 0.20,
        f'Media global = {df_clean["puntos"].mean():.2f}\n'
        f'Varianza global = {df_clean["puntos"].var():.2f}\n'
        f'Razon Var/Media = {ratio_global:.4f}',
        transform=ax.transAxes, fontsize=10, verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

plt.tight_layout()
plt.savefig('plots/media_varianza_temporada.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nTabla de media y varianza por temporada:')
print(agg.to_string())


Tabla de media y varianza por temporada:
                 mean         var  count
temporada                               
2023-24    117.768293  137.414785     82
2024-25    113.792683  182.931798     82


---
## 13. Poisson No Homogéneo: Local vs Visitante

El modelo Poisson homogéneo asume un solo $\lambda$. Comparamos si un modelo con $\lambda$ distinto para partidos de local y visitante ajusta mejor (usando AIC como criterio).

In [12]:
no_homogeneo = analizar_poisson_no_homogeneo(df_clean)

if no_homogeneo['aic_homogeneo'] is not None:
    print(f'lambda global:               {no_homogeneo["lambda_global"]:.2f}')
    print(f'lambda local (casa):         {no_homogeneo["lambda_local"]:.2f}')
    print(f'lambda visitante (fuera):    {no_homogeneo["lambda_visitante"]:.2f}')
    print(f'Partidos local:              {no_homogeneo["n_local"]}')
    print(f'Partidos visitante:          {no_homogeneo["n_visitante"]}')
    print(f'AIC homogeneo:               {no_homogeneo["aic_homogeneo"]:.2f}')
    print(f'AIC no homogeneo:            {no_homogeneo["aic_no_homogeneo"]:.2f}')
    print(f'\nConclusion: {no_homogeneo["conclusion"]}')

    fig, ax = plt.subplots(figsize=(8, 5))
    categorias = ['Global', 'Local (casa)', 'Visitante (fuera)']
    valores_lambda = [
        no_homogeneo['lambda_global'],
        no_homogeneo['lambda_local'],
        no_homogeneo['lambda_visitante']
    ]
    colores = ['#607D8B', '#2196F3', '#FF9800']

    bars = ax.bar(categorias, valores_lambda, color=colores, edgecolor='white', linewidth=1.5, width=0.5)
    for bar, val in zip(bars, valores_lambda):
        if val is not None:
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                    f'{val:.1f}', ha='center', va='bottom', fontweight='bold', fontsize=13)

    ax.set_ylabel('$\\lambda$ (puntos esperados)')
    ax.set_title(f'Poisson Homogeneo vs No Homogeneo \u2014 {NOMBRE_EQUIPO}')

    ax.text(0.98, 0.95,
            f'AIC homogeneo = {no_homogeneo["aic_homogeneo"]:.1f}\n'
            f'AIC no homogeneo = {no_homogeneo["aic_no_homogeneo"]:.1f}\n'
            f'Menor AIC = mejor modelo',
            transform=ax.transAxes, fontsize=10, verticalalignment='top',
            horizontalalignment='right',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.85))

    plt.tight_layout()
    plt.savefig('plots/poisson_no_homogeneo.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('Sin datos de local/visitante para el analisis.')

lambda global:               115.78
lambda local (casa):         114.99
lambda visitante (fuera):    116.57
Partidos local:              82
Partidos visitante:          82
AIC homogeneo:               1313.23
AIC no homogeneo:            1314.34

Conclusion: El modelo HOMOGENEO tiene mejor AIC. No hay evidencia suficiente de que lambda varie por local/visitante.


---
## 14. Tiempos entre Eventos y Distribución Exponencial

Definimos un "evento" como un partido donde los puntos superan $\lambda + \sigma$ (media + 1 desviación).

Si el proceso subyacente es Poisson, los tiempos entre eventos consecutivos deben seguir una distribución exponencial: $T \sim \text{Exp}(\lambda_{exp})$, con $E[T] = 1/\lambda_{exp}$.

In [13]:
tiempos, umbral, n_eventos = identificar_eventos_y_tiempos(puntos, lambda_est)

print(f'Umbral de evento (lambda + sigma): {umbral:.1f} puntos')
print(f'Numero de eventos detectados:      {n_eventos}')

if len(tiempos) >= 2:
    lambda_exp = 1.0 / np.mean(tiempos)
    print(f'Media de tiempos entre eventos:    {np.mean(tiempos):.2f} partidos')
    print(f'Lambda exponencial estimado:       {lambda_exp:.4f}')
    print(f'Tiempo esperado (1/lambda_exp):    {1.0/lambda_exp:.2f} partidos')

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    ax = axes[0]
    ax.hist(tiempos, bins=20, density=True, alpha=0.7, color='#4CAF50',
            edgecolor='white', linewidth=0.5,
            label=f'Tiempos observados (n={len(tiempos)})')
    x_exp = np.linspace(0, max(tiempos) * 1.2, 200)
    ax.plot(x_exp, stats.expon.pdf(x_exp, scale=1.0/lambda_exp),
            '-', color='#D32F2F', linewidth=2.5,
            label=f'Exponencial teorica ($\\lambda$={lambda_exp:.4f})')
    ax.axvline(np.mean(tiempos), color='#FF9800', linestyle='--', linewidth=2,
               label=f'Media obs = {np.mean(tiempos):.2f}')
    ax.set_xlabel('Partidos entre eventos')
    ax.set_ylabel('Densidad')
    ax.set_title(f'Distribucion de Tiempos entre Eventos')
    ax.legend(fontsize=9)

    ax = axes[1]
    t_sorted = np.sort(tiempos)
    n_t = len(tiempos)
    theo_exp = stats.expon.ppf((np.arange(1, n_t + 1) - 0.5) / n_t,
                                scale=1.0/lambda_exp)
    ax.scatter(theo_exp, t_sorted, alpha=0.5, color='#4CAF50',
               edgecolors='white', linewidth=0.3, s=50)
    ax.plot(theo_exp, theo_exp, '--', color='#D32F2F', linewidth=2,
            label='Linea de referencia (y = x)')
    ax.set_xlabel('Cuantiles teoricos (Exponencial)')
    ax.set_ylabel('Cuantiles observados')
    ax.set_title(f'Q-Q Plot Exponencial')
    ax.legend(loc='upper left')

    ax.text(0.98, 0.10,
            f'$\\lambda_{{exp}}$ = {lambda_exp:.4f}\n'
            f'media obs = {np.mean(tiempos):.2f}\n'
            f'n = {n_t} eventos',
            transform=ax.transAxes, fontsize=9, verticalalignment='top',
            horizontalalignment='right',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

    plt.tight_layout()
    plt.savefig('plots/tiempos_exponencial.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('Muy pocos eventos para analisis exponencial.')

Umbral de evento (lambda + sigma): 128.5 puntos
Numero de eventos detectados:      27
Media de tiempos entre eventos:    6.04 partidos
Lambda exponencial estimado:       0.1656
Tiempo esperado (1/lambda_exp):    6.04 partidos


---
## 15. Cadenas de Markov: Resultados W/L

Modelamos la secuencia de victorias (W) y derrotas (L) como una cadena de Markov de 2 estados. La matriz de transición $P$ contiene:

$$P = \begin{bmatrix} P(W \mid W) & P(L \mid W) \\ P(W \mid L) & P(L \mid L) \end{bmatrix}$$

La distribución estacionaria $\pi$ satisface $\pi = \pi P$ y representa la proporción de victorias/derrotas a largo plazo.

In [14]:
resultados = df_clean['resultado'].values

P, labels, conteos = construir_matriz_transicion(resultados)
pi = distribucion_estacionaria(P)
prediccion = predecir_siguiente(resultados, P)

print('--- Matriz de Transicion ---')
print(f'  P(W|W) = {P[0,0]:.3f}    P(L|W) = {P[0,1]:.3f}')
print(f'  P(W|L) = {P[1,0]:.3f}    P(L|L) = {P[1,1]:.3f}')
print()
print('--- Conteos de Transiciones ---')
print(f'  W -> W: {conteos["W->W"]}    W -> L: {conteos["W->L"]}')
print(f'  L -> W: {conteos["L->W"]}    L -> L: {conteos["L->L"]}')
print()
print('--- Distribucion Estacionaria ---')
print(f'  pi_W (victorias a largo plazo) = {pi[0]:.3f}')
print(f'  pi_L (derrotas a largo plazo)   = {pi[1]:.3f}')
print(f'  Proporcion observada de W:       {np.mean(resultados == "W"):.3f}')
print()
print(f'--- Prediccion Siguiente Partido ---')
print(f'  Ultimo resultado: {prediccion["ultimo_resultado"]}')
print(f'  P(siguiente = W) = {prediccion["probabilidad_W"]:.3f}')
print(f'  P(siguiente = L) = {prediccion["probabilidad_L"]:.3f}')
print(f'  Prediccion: {prediccion["prediccion"]}')

fig, ax = plt.subplots(figsize=(9, 7))

r = 3
t_W = np.pi / 2
t_L = 3 * np.pi / 2
pos = {'W': (r * np.cos(t_W), r * np.sin(t_W)),
       'L': (r * np.cos(t_L), r * np.sin(t_L))}

for label, (x, y) in pos.items():
    circle = plt.Circle((x, y), 0.65, color='#2196F3', ec='white',
                         linewidth=2, zorder=3)
    ax.add_patch(circle)
    ax.text(x, y, label, ha='center', va='center', fontsize=20,
            fontweight='bold', color='white', zorder=4)

for origen, color in [('W', '#4CAF50'), ('L', '#FF9800')]:
    idx_origen = 0 if origen == 'W' else 1
    idx_opuesto = 1 if origen == 'W' else 0
    x0, y0 = pos[origen]
    x1, y1 = pos['L' if origen == 'W' else 'W']
    dx, dy = x1 - x0, y1 - y0
    angle = np.arctan2(dy, dx)
    x0_adj = x0 + 0.65 * np.cos(angle)
    y0_adj = y0 + 0.65 * np.sin(angle)
    x1_adj = x1 - 0.65 * np.cos(angle)
    y1_adj = y1 - 0.65 * np.sin(angle)

    ax.annotate('', xy=(x1_adj, y1_adj), xytext=(x0_adj, y0_adj),
                 arrowprops=dict(arrowstyle='->', color=color, lw=2.5,
                                 connectionstyle='arc3,rad=.2'))

    prob = P[idx_origen, idx_opuesto]
    mid_x = (x0 + x1) / 2 + 0.8 * np.cos(angle + np.pi/2)
    mid_y = (y0 + y1) / 2 + 0.8 * np.sin(angle + np.pi/2)
    ax.text(mid_x, mid_y, f'{prob:.3f}', fontsize=13, ha='center', va='center',
            bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.85))

    prob_self = P[idx_origen, idx_origen]
    angle_self = np.pi/4 if origen == 'W' else -np.pi/4
    loop_r = 1.0
    cx = x0 + (0.65 + loop_r) * np.cos(angle_self)
    cy = y0 + (0.65 + loop_r) * np.sin(angle_self)
    theta = np.linspace(angle_self - 1.0, angle_self + 1.0, 60)
    lx = x0 + 0.65 * np.cos(theta)
    ly = y0 + 0.65 * np.sin(theta)
    ax.plot(lx, ly, color='#D32F2F', linewidth=2.2, alpha=0.7)
    end_idx = -1
    ax.annotate('', xy=(lx[end_idx], ly[end_idx]),
                xytext=(lx[end_idx-1], ly[end_idx-1]),
                arrowprops=dict(arrowstyle='->', color='#D32F2F', lw=2.2))
    tx = x0 + (0.65 + loop_r + 0.4) * np.cos(angle_self)
    ty = y0 + (0.65 + loop_r + 0.4) * np.sin(angle_self)
    ax.text(tx, ty, f'{prob_self:.3f}', fontsize=12, ha='center', va='center',
            bbox=dict(boxstyle='round,pad=0.15', facecolor='white', alpha=0.85))

ax.set_xlim(-7, 7)
ax.set_ylim(-5.5, 5.5)
ax.set_aspect('equal')
ax.set_title(f'Cadena de Markov: Resultados W/L \u2014 {NOMBRE_EQUIPO}')
ax.axis('off')

ax.text(0.02, 0.97,
        'Diagrama de transicion de la cadena de Markov.\n'
        'Muestra las probabilidades de pasar de un estado\n'
        '(W=Victoria, L=Derrota) al siguiente.',
        transform=ax.transAxes, fontsize=10, verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.85))

plt.tight_layout()
plt.savefig('plots/markov_transiciones_nb.png', dpi=150, bbox_inches='tight')
plt.show()

--- Matriz de Transicion ---
  P(W|W) = 0.553    P(L|W) = 0.447
  P(W|L) = 0.609    P(L|L) = 0.391

--- Conteos de Transiciones ---
  W -> W: 52    W -> L: 42
  L -> W: 42    L -> L: 27

--- Distribucion Estacionaria ---
  pi_W (victorias a largo plazo) = 0.577
  pi_L (derrotas a largo plazo)   = 0.423
  Proporcion observada de W:       0.573

--- Prediccion Siguiente Partido ---
  Ultimo resultado: L
  P(siguiente = W) = 0.609
  P(siguiente = L) = 0.391
  Prediccion: W


---
## 16. Resumen Final y Conclusiones

Recopilamos todos los resultados en una tabla resumen y emitimos la conclusión final sobre los tres modelos.

In [15]:
resumen = resumen_estadistico(stats_dict, propiedad,
                              estacionariedad=estacionariedad,
                              no_homogeneo=no_homogeneo)
imprimir_resumen(resumen)

summary_data = {
    'Metrica': [
        'Partidos (n)', 'Media (lambda)', 'Varianza',
        'Desv. Estandar', 'Minimo', 'Maximo', 'Mediana',
        'Asimetria', 'Curtosis', 'Razon Var/Media',
        'ADF estadistico', 'ADF p-valor',
        'KPSS estadistico', 'KPSS p-valor',
        'lambda local', 'lambda visitante',
        'AIC homogeneo', 'AIC no homogeneo',
        'P(W|W)', 'P(L|L)',
        'pi_W estacionaria', 'Proporcion W observada'
    ],
    'Valor': [
        stats_dict['n'], f"{stats_dict['media']:.2f}", f"{stats_dict['varianza']:.2f}",
        f"{stats_dict['desviacion_estandar']:.2f}", f"{stats_dict['minimo']:.0f}",
        f"{stats_dict['maximo']:.0f}", f"{stats_dict['mediana']:.0f}",
        f"{stats_dict['asimetria']:.3f}", f"{stats_dict['curtosis']:.3f}",
        f"{propiedad['ratio_varianza_media']:.4f}",
        f"{estacionariedad['adf_estadistico']:.3f}" if estacionariedad['adf_estadistico'] else 'N/A',
        f"{estacionariedad['adf_p_valor']:.4f}" if estacionariedad['adf_p_valor'] else 'N/A',
        f"{estacionariedad['kpss_estadistico']:.3f}" if estacionariedad['kpss_estadistico'] else 'N/A',
        f"{estacionariedad['kpss_p_valor']:.4f}" if estacionariedad['kpss_p_valor'] else 'N/A',
        f"{no_homogeneo['lambda_local']:.2f}" if no_homogeneo['lambda_local'] else 'N/A',
        f"{no_homogeneo['lambda_visitante']:.2f}" if no_homogeneo['lambda_visitante'] else 'N/A',
        f"{no_homogeneo['aic_homogeneo']:.2f}" if no_homogeneo['aic_homogeneo'] else 'N/A',
        f"{no_homogeneo['aic_no_homogeneo']:.2f}" if no_homogeneo['aic_no_homogeneo'] else 'N/A',
        f'{P[0,0]:.3f}', f'{P[1,1]:.3f}',
        f'{pi[0]:.3f}', f'{np.mean(resultados == "W"):.3f}'
    ]
}

df_summary = pd.DataFrame(summary_data)
print('\nTabla resumen:')
print(df_summary.to_string(index=False))

os.makedirs('data/processed', exist_ok=True)
df_summary.to_csv('data/processed/resumen_completo.csv', index=False)
print('\nResumen guardado en: data/processed/resumen_completo.csv')


  RESUMEN DEL ANALISIS ESTADISTICO

  --- Estadistica Descriptiva ---
  Partidos analizados (n):    164
  Media muestral (lambda):    115.78
  Varianza muestral:          162.17
  Desviacion estandar:        12.73
  Minimo:                     85
  Maximo:                     148
  Asimetria:                  -0.044
  Curtosis:                   -0.413

  --- Propiedad E[X] = Var(X) = lambda ---
  Razon Var(X)/E[X]:          1.4007
  Diferencia relativa:        0.4007
  Diagnostico:                DESVIACIÓN SIGNIFICATIVA: Los datos muestran sobredispersión. Poisson puede no ser adecuado.

  --- Pruebas de Estacionariedad ---
  ADF estadistico:            -3.9835
  ADF p-valor:                0.0015
  ADF conclusion:             Serie ESTACIONARIA (rechaza H0 de raiz unitaria)
  KPSS estadistico:           0.4442
  KPSS p-valor:               0.0581
  KPSS conclusion:            Serie ESTACIONARIA (no rechaza H0)

  --- Poisson Homogeneo vs No Homogeneo ---
  lambda global:           

---
## 17. Interpretación Matemática

### Modelo Poisson
Si $X \sim \text{Poisson}(\lambda)$, donde $X$ son los puntos anotados:
- $\lambda$ es la tasa promedio de anotación por partido.
- $P(X = k) = \frac{\lambda^k e^{-\lambda}}{k!}$.
- Intervalo de predicción (~95%): $[\lambda - 2\sqrt{\lambda},\; \lambda + 2\sqrt{\lambda}]$.

### Modelo Exponencial
Si los eventos de altos puntos ocurren según Poisson, los tiempos $T$ entre ellos siguen:
- $T \sim \text{Exp}(\lambda_{exp})$, con $f(t) = \lambda_{exp} e^{-\lambda_{exp} t}$.
- $E[T] = 1/\lambda_{exp}$ (partidos promedio entre eventos).

### Cadena de Markov
La matriz de transición $P$ captura la dinámica de rachas:
- $P(W \mid L)$ mide la capacidad de recuperación tras una derrota.
- $P(L \mid L)$ mide la tendencia a rachas perdedoras.
- $\pi_W$ es la proporción de victorias a largo plazo.

### Estacionariedad
- ADF (p < 0.05): la serie no tiene raíz unitaria → estacionaria.
- KPSS (p ≥ 0.05): no se rechaza que sea estacionaria.
- Ambas pruebas consistentes → $\lambda$ es estable en el tiempo.

### Limitaciones
- Poisson asume independencia y tasa constante (sobredispersión es común).
- Markov de 2 estados ignora empates y magnitud de la diferencia.
- Los datos de 2 temporadas pueden no capturar ciclos más largos.

In [16]:
print(f'Intervalo del 95% aproximado:')
low = lambda_est - 2 * np.sqrt(lambda_est)
high = lambda_est + 2 * np.sqrt(lambda_est)
print(f'  [{low:.1f}, {high:.1f}] puntos por partido')

print(f'\nProbabilidades segun el modelo Poisson:')
k_medio = int(lambda_est)
print(f'  P(X = {k_medio}) = {poisson.pmf(k_medio, mu=lambda_est):.4f}')
k_bajo = int(lambda_est - np.sqrt(lambda_est))
print(f'  P(X < {k_bajo}) = {poisson.cdf(k_bajo, mu=lambda_est):.4f}')
k_alto = int(lambda_est + np.sqrt(lambda_est))
print(f'  P(X > {k_alto}) = {1 - poisson.cdf(k_alto, mu=lambda_est):.4f}')

if len(tiempos) >= 2:
    print(f'\nModelo exponencial (tiempos entre eventos):')
    print(f'  Tiempo promedio entre eventos de altos puntos: {1.0/lambda_exp:.2f} partidos')
    print(f'  P(T > 5 partidos sin evento) = {np.exp(-lambda_exp * 5):.4f}')

print(f'\nCadena de Markov:')
print(f'  Probabilidad de ganar tras perder: {P[1,0]:.3f}')
print(f'  Probabilidad de ganar tras ganar:  {P[0,0]:.3f}')
print(f'  Rachas: el equipo gana mas seguido tras una derrota que tras una victoria' if P[1,0] > P[0,0] else f'  Rachas: el equipo gana mas seguido tras una victoria')

Intervalo del 95% aproximado:
  [94.3, 137.3] puntos por partido

Probabilidades segun el modelo Poisson:
  P(X = 115) = 0.0371
  P(X < 105) = 0.1699
  P(X > 126) = 0.1595

Modelo exponencial (tiempos entre eventos):
  Tiempo promedio entre eventos de altos puntos: 6.04 partidos
  P(T > 5 partidos sin evento) = 0.4369

Cadena de Markov:
  Probabilidad de ganar tras perder: 0.609
  Probabilidad de ganar tras ganar:  0.553
  Rachas: el equipo gana mas seguido tras una derrota que tras una victoria


---
## 18. Referencias

1. Ross, S. M. (2014). *Introduction to Probability Models*. Academic Press.
2. Taylor, H. M., & Karlin, S. (1998). *An Introduction to Stochastic Modeling*. Academic Press.
3. nba_api Documentation. (2024). https://github.com/swar/nba_api
4. NBA.com Official Statistics. https://www.nba.com/stats
5. Law, A. M. (2015). *Simulation Modeling and Analysis*. McGraw-Hill.
6. D'Agostino, R. B., & Stephens, M. A. (1986). *Goodness-of-Fit Techniques*. CRC Press.
7. Seabold, S., & Perktold, J. (2010). *Statsmodels: Econometric and Statistical Modeling with Python.*

---

*Proyecto desarrollado para la materia de Procesos Estocasticos.*
*Temas implementados: Proceso de Poisson, Distribucion Exponencial, Cadenas de Markov, Estacionariedad.*